# Member 2 | Portfolio construction and pre-2023 checks

Case decision: 31 December 2022. **Input vintage:** retrieved retrospectively from Yahoo in October 2026; no post-2022 performance is used to choose this policy. Run `python src/rebuild.py` to reproduce the raw-to-processed data pipeline before opening the notebook. These notebook cells use only the freshly rebuilt estimation outputs.

This is a Member 2 notebook, not a completed 2023-2025 backtest or French factor regression.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = Path.cwd() if (Path.cwd()/"src/rebuild.py").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/"src"))
from portfolio_utils import ASSETS,MARKET,parameters,optimize,measures,F
p=ROOT/"data/processed/asset_monthly_returns_estimation.csv"
r=pd.read_csv(p)
assert len(r)==96 and r.month.iloc[0]=='2015-01' and r.month.iloc[-1]=='2022-12'
print(f"Verified {len(r)} monthly observations, {len(ASSETS)} assets and VTI, information cutoff 2022-12.")

Verified 96 monthly observations, 12 assets and VTI, information cutoff 2022-12.


## Forecast, sample risk and single-index risk

Forecast `E[r_i] = 12 × [0.25 × historical monthly mean_i + 0.75 × historical monthly VTI mean]`. The two covariance models use exactly the same observation dates and securities. The single-index model assumes zero correlations between idiosyncratic residuals; this assumption can be checked rather than presumed true.

In [2]:
mu,S,SI,beta,residuals=parameters(r)
print('Largest absolute annual risk covariance difference:',round(float(np.max(np.abs(S-SI))),6))
print('Largest off-diagonal residual correlation:',round(float(np.max(np.abs(np.corrcoef(residuals,rowvar=False)-np.eye(12)))),4))

Largest absolute annual risk covariance difference: 0.026511
Largest off-diagonal residual correlation: 0.7951


## Portfolio competition, common caps

Long-only; individual common stock at most 15%; direct-stock sector at most 30%; any single ETF at most 30%, combined ETFs at most 40%. The model uses foundation risk aversion A=3 with equity-notional scaling; A=6 is reported separately. No constraint forces a positive ETF weight.

In [3]:
w_mv=optimize(mu,S,3)
w_si=optimize(mu,SI,3)
w_min=optimize(mu,S,3,minvar=True)
w_a6=optimize(mu,S,6)
W=pd.DataFrame({'ticker':ASSETS,'Sample_MV_A3':w_mv,'Single_Index_A3':w_si,'Minimum_Variance':w_min,'Sample_MV_A6':w_a6})
print(W.to_string(index=False,float_format=lambda x:f'{x:.5f}'))
print(pd.DataFrame({k:measures(v,mu,S,A=6 if 'A6' in k else 3) for k,v in [('Sample_MV_A3',w_mv),('Single_Index_A3',w_si),('Minimum_Variance',w_min),('Sample_MV_A6',w_a6)]}).T.to_string(float_format=lambda x:f'{x:.6f}'))

ticker  Sample_MV_A3  Single_Index_A3  Minimum_Variance  Sample_MV_A6
  AAPL       0.15000          0.15000           0.00000       0.13945
  MSFT       0.15000          0.15000           0.09790       0.15000
   JPM       0.04242          0.00000           0.00000       0.04391
   BAC       0.00000          0.00000           0.00000       0.00000
   JNJ       0.09098          0.15000           0.15000       0.15000
   PFE       0.14171          0.13725           0.10694       0.13937
    PG       0.15000          0.15000           0.15000       0.15000
    KO       0.12593          0.15000           0.15000       0.15000
   XOM       0.00000          0.00000           0.00000       0.00018
   CAT       0.14897          0.11275           0.00000       0.07709
   VTV       0.00000          0.00000           0.30000       0.00000
   IWM       0.00000          0.00000           0.04516       0.00000
                  annual_expected_arithmetic_return  annual_volatility  equity_component_o

## Independent decision and limitations

The recommended Sample MV reaches the highest estimated sample-covariance utility. This does not establish out-of-sample superiority. The comparison is retrospective; equal-weight, single-index and constrained minimum variance are genuine challengers. Member 3 receives the historical **static-exposure portfolio return proxy**, not the actual 2023-2025 buy-and-hold performance. No fictional case-bond market return series is created.

In [4]:
w_source=pd.read_csv(ROOT/'evidence/decision_weights_confirmed.csv').set_index('security').loc[ASSETS]['weight_of_growth_sleeve'].to_numpy(float)
assert np.max(np.abs(w_source-w_mv))<2e-7
ref=pd.read_csv(ROOT/'outputs/fixed_weight_reversal.csv')
print('Recorded chosen weights match rebuilt weights. Sensitivity thresholds:')
print(ref[['test','threshold']].to_string(index=False))
print('Member 3 handoff:',ROOT/'outputs/growth_static_exposure_returns.csv')

Recorded chosen weights match rebuilt weights. Sensitivity thresholds:
                                                   test  threshold
Historical-mean prior weight lambda, two fixed policies   0.084171
         Foundation risk aversion A, two fixed policies   8.910441
Member 3 handoff: /mnt/data/Kappa_M2_Complete/outputs/growth_static_exposure_returns.csv
